# RAG

## 1. LLM

Завантажуємо АРІ-ключ з файлу .env у змінну оточення GEMINI_API_KEY.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

Створюємо клієнт Gemini та функцію для виклику моделі.

In [2]:
from google import genai

client = genai.Client()

In [3]:

def llm(prompt):
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )
    return response.text

Перевіримо, як все працює. Зробимо запит до моделі і отримаємо відповідь.

In [4]:
question = "I just discovered the course. Can I join now?"

In [ ]:
answer = llm(question)
print(answer)

To give you the most accurate answer, **could you tell me a little more about the course?** (e.g., the name of the course or the platform/institution offering it).

In the meantime, here are a few general things you should check:

1.  **Check the Enrollment Status:** Go to the course website or dashboard. Does it say "Open for Enrollment," "Join Now," or is it "Closed/Archived"?
2.  **Check for "Self-Paced" vs. "Cohort-Based":**
    *   **Self-Paced:** These usually allow you to join at any time and work through the material on your own schedule.
    *   **Cohort-Based:** These have specific start and end dates. If the course started a few weeks ago, you might have missed too much content to catch up, though some allow late registration.
3.  **Check the Syllabus/Schedule:** If it is a live or cohort-based course, see if there are assignments or live sessions you have already missed. If the course is mostly recorded videos, you can usually jump in right away.
4.  **Look for a Contact Em

Ми отримали закономірну відповідь. Нейронка нічого не знає про курс. Треба надати їй додаткову інформацію, яку ми візьмемо з FAQ курсу.

In [5]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

Створимо промпт

In [6]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

Знову викличемо модель та передамо їй промпт.

In [7]:
answer = llm(prompt)
print(answer)

Yes, you can join now, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


Отримали цілком конкретну відповідь.

## 2. Database

### Отримання даних

В якості бази даних для LLM будемо використовувати FAQ курсів Data Talks Club. Вони оброблені і готові до використання та доступні на сайті в форматі JSON. Отримаємо їх.

In [2]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

Цей код поверне нам список курсів від Data Talks Club. Виведемо його і подивимось. 

In [3]:
import json
print(json.dumps(courses_raw, ensure_ascii=False, indent=2))

[
  {
    "course": "data-engineering-zoomcamp",
    "course_name": "Data Engineering Zoomcamp",
    "path": "/json/data-engineering-zoomcamp.json",
    "questions_count": 404
  },
  {
    "course": "stock-markets-analytics-zoomcamp",
    "course_name": "Stock Markets Analytics Zoomcamp",
    "path": "/json/stock-markets-analytics-zoomcamp.json",
    "questions_count": 93
  },
  {
    "course": "ai-dev-tools-zoomcamp",
    "course_name": "AI Dev Tools Zoomcamp",
    "path": "/json/ai-dev-tools-zoomcamp.json",
    "questions_count": 41
  },
  {
    "course": "llm-zoomcamp",
    "course_name": "LLM Zoomcamp",
    "path": "/json/llm-zoomcamp.json",
    "questions_count": 85
  },
  {
    "course": "mlops-zoomcamp",
    "course_name": "MLOps Zoomcamp",
    "path": "/json/mlops-zoomcamp.json",
    "questions_count": 255
  },
  {
    "course": "machine-learning-zoomcamp",
    "course_name": "ML Zoomcamp",
    "path": "/json/machine-learning-zoomcamp.json",
    "questions_count": 472
  }
]


Кожний запис має поле PATH, що містить шлях до FAQу конкретного курсу. Отримаємо тепер їх усі.

In [4]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

Ось що відбувається у коді в попередній комірці.

1. Функція requests.get() надсилає запит на вказану адресу (course_url), щоб отримати дані (зазвичай у форматі JSON). Результатом виконання є об'єкт відповіді (Response), який зберігається у змінну course_response. Цей об'єкт містить статус відповіді, заголовки, тіло відповіді тощо.

2. Метод .raise_for_status() перевіряє статус HTTP-відповіді. Якщо запит успішний (статус 200–299), нічого не відбувається. Якщо сталася помилка (наприклад, 404 Not Found або 500 Server Error), цей метод автоматично викликає виключення (exception) HTTPError. Це зупиняє подальше виконання коду, якщо дані не були отримані успішно, запобігаючи використанню "битих" даних.

3. Оскільки дані з API приходять як рядок у форматі JSON, метод .json() перетворює цей рядок на зрозумілу для Python структуру даних (список або словник). Тепер course_data — це готовий до роботи об'єкт Python.

4. Метод .extend() бере всі елементи з course_data (який є списком) і додає кожен з них до списку documents як окремий елемент.

Подивимось, як виглядають окремі елементи нашого списку.

In [5]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

### Індексація